In [ ]:
import importlib.util
import subprocess
import sys

required_packages = {
    "lightgbm": "lightgbm",
    "xgboost": "xgboost",
    "seaborn": "seaborn",
}

for module_name, package_name in required_packages.items():
    if importlib.util.find_spec(module_name) is None:
        print(f"Installing {package_name}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])

print("Package check complete.")

In [ ]:
import gc
import os
import re
import shutil
import warnings
import zipfile
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Tuple

import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.calibration import calibration_curve
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    matthews_corrcoef,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
)
from xgboost import XGBClassifier

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

SEED = 42

# Start with True to verify that everything runs.
# Change to False for the complete experiment.
DEMO_MODE = True

DEMO_ROWS = {
    "train": 80_000,
    "validation": 20_000,
    "test": 20_000,
}

BOOTSTRAP_REPEATS = 500 if DEMO_MODE else 5_000

KAGGLE_INPUT = Path("/kaggle/input")
WORKING_DIR = Path("/kaggle/working")
EXTRACT_DIR = WORKING_DIR / "household_extracted"
OUTPUT_DIR = WORKING_DIR / "household_peak_outputs"

EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Kaggle input exists:", KAGGLE_INPUT.exists())
print("Demo mode:", DEMO_MODE)
print("Output directory:", OUTPUT_DIR)

## Cell 3 — Find the `household data` folder and `sensordata.zip`

In [ ]:
def find_prepared_dataset(input_root=KAGGLE_INPUT):
    exact_matches = sorted(
        input_root.glob('**/supervised_95_features.csv.gz')
    )

    if exact_matches:
        return exact_matches[0]

    flexible_matches = sorted(
        path for path in input_root.glob('**/*')
        if path.is_file()
        and 'supervised' in path.name.lower()
        and path.name.lower().endswith(('.csv', '.csv.gz', '.parquet'))
    )

    if flexible_matches:
        return flexible_matches[0]

    raise FileNotFoundError(
        'Could not find supervised_95_features.csv.gz under /kaggle/input. '
        'Use Add Input to attach the dataset containing this file.'
    )


PREPARED_DATA_FILE = find_prepared_dataset()
print('Prepared modelling and weather file:')
print(PREPARED_DATA_FILE)
print(f'Compressed size: {PREPARED_DATA_FILE.stat().st_size / (1024**2):.2f} MB')

## Cell 6 — Inventory all sensor and weather file

In [ ]:
TABULAR_SUFFIXES = (
    ".csv", ".csv.gz", ".txt", ".tsv", ".parquet",
    ".feather", ".json", ".xlsx"
)

SEARCH_ROOTS = list(HOUSEHOLD_ROOTS)
if EXTRACT_DIR.exists():
    SEARCH_ROOTS.append(EXTRACT_DIR)

all_tabular_files = find_files(
    SEARCH_ROOTS,
    suffixes=TABULAR_SUFFIXES,
)

weather_files = [
    p for p in all_tabular_files
    if any(term in normalise_name(str(p)) for term in (
        "weather", "temperature", "humidity",
        "precipitation", "windspeed", "metoffice"
    ))
]

sensor_files = [
    p for p in all_tabular_files
    if any(term in normalise_name(str(p)) for term in (
        "sensor", "electric", "electricity", "power",
        "energy", "mains", "appliance"
    ))
]

inventory = pd.DataFrame({
    "path": [str(p) for p in all_tabular_files],
    "filename": [p.name for p in all_tabular_files],
    "suffix": ["".join(p.suffixes).lower() for p in all_tabular_files],
    "size_mb": [p.stat().st_size / (1024 ** 2) for p in all_tabular_files],
})

inventory = inventory.sort_values("size_mb", ascending=False).reset_index(drop=True)

print(f"All tabular files: {len(all_tabular_files):,}")
print(f"Likely sensor files: {len(sensor_files):,}")
print(f"Likely weather files: {len(weather_files):,}")

display(inventory.head(50))

print("\nLikely weather files:")
for path in weather_files[:50]:
    print(" -", path)

## Cell 7 — Safely inspect columns without loading entire large files


In [ ]:
def read_preview(path: Path, nrows: int = 5) -> pd.DataFrame:
    name = path.name.lower()

    if name.endswith(".parquet"):
        return pd.read_parquet(path).head(nrows)

    if name.endswith(".feather"):
        return pd.read_feather(path).head(nrows)

    if name.endswith(".xlsx"):
        return pd.read_excel(path, nrows=nrows)

    if name.endswith(".json"):
        return pd.read_json(path).head(nrows)

    # Try common separators for text files.
    attempts = [
        {"sep": ","},
        {"sep": "\t"},
        {"sep": ";"},
        {"sep": None, "engine": "python"},
    ]

    last_error = None

    for options in attempts:
        try:
            return pd.read_csv(
                path,
                nrows=nrows,
                low_memory=False,
                **options,
            )
        except Exception as error:
            last_error = error

    raise RuntimeError(f"Could not preview {path}: {last_error}")


def inspect_files(paths: List[Path], limit: int = 20) -> pd.DataFrame:
    rows = []

    for path in paths[:limit]:
        try:
            preview = read_preview(path, nrows=5)
            rows.append({
                "path": str(path),
                "size_mb": path.stat().st_size / (1024 ** 2),
                "rows_previewed": len(preview),
                "column_count": len(preview.columns),
                "columns": list(preview.columns),
                "status": "OK",
            })
        except Exception as error:
            rows.append({
                "path": str(path),
                "size_mb": path.stat().st_size / (1024 ** 2),
                "rows_previewed": 0,
                "column_count": 0,
                "columns": [],
                "status": f"ERROR: {error}",
            })

    return pd.DataFrame(rows)


print("Sensor-file inspection:")
sensor_inspection = inspect_files(sensor_files, limit=25)
display(sensor_inspection)

print("\nWeather-file inspection:")
weather_inspection = inspect_files(weather_files, limit=25)
display(weather_inspection)

## Weather data checks inside the prepared CSV


In [ ]:
WEATHER_CHECK_COLUMNS = [
    'origin_timestamp_utc',
    'location',
    'origin_temperature_2m_c',
    'origin_relative_humidity_2m_pct',
    'origin_precipitation_mm',
    'origin_wind_speed_10m_kmh',
    'temperature_lag_24h_c',
    'humidity_lag_24h_pct',
    'precipitation_lag_24h_mm',
    'wind_speed_lag_24h_kmh',
]

weather_data = pd.read_csv(
    PREPARED_DATA_FILE,
    usecols=WEATHER_CHECK_COLUMNS,
    parse_dates=['origin_timestamp_utc'],
    low_memory=False,
)

print(f'Weather rows: {len(weather_data):,}')
print(
    'Weather period:',
    weather_data['origin_timestamp_utc'].min(),
    'to',
    weather_data['origin_timestamp_utc'].max(),
)
print('\nLocations:')
display(weather_data['location'].value_counts(dropna=False).to_frame('rows'))

weather_numeric = WEATHER_CHECK_COLUMNS[2:]

weather_quality = pd.DataFrame({
    'missing_rows': weather_data[weather_numeric].isna().sum(),
    'missing_percent': weather_data[weather_numeric].isna().mean() * 100,
    'minimum': weather_data[weather_numeric].min(),
    'mean': weather_data[weather_numeric].mean(),
    'maximum': weather_data[weather_numeric].max(),
})

display(weather_quality.round(3))

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
plot_columns = [
    'origin_temperature_2m_c',
    'origin_relative_humidity_2m_pct',
    'origin_precipitation_mm',
    'origin_wind_speed_10m_kmh',
]

for axis, column in zip(axes.ravel(), plot_columns):
    sns.histplot(weather_data[column], bins=40, ax=axis)
    axis.set_title(column.replace('_', ' ').title())

plt.tight_layout()
plt.show()

del weather_data
gc.collect()


## Prepared file validation


In [ ]:
LOAD_FEATURES = [
    'load_lag_0h_w', 'load_lag_1h_w', 'load_lag_2h_w',
    'load_lag_3h_w', 'load_lag_6h_w', 'load_lag_12h_w',
    'load_lag_24h_w', 'load_lag_48h_w',
    'load_roll_24h_mean_w', 'load_roll_24h_std_w',
    'load_roll_168h_mean_w', 'load_roll_168h_std_w',
]

WEATHER_FEATURES = [
    'origin_temperature_2m_c',
    'origin_relative_humidity_2m_pct',
    'origin_precipitation_mm',
    'origin_wind_speed_10m_kmh',
    'temperature_lag_24h_c',
    'humidity_lag_24h_pct',
    'precipitation_lag_24h_mm',
    'wind_speed_lag_24h_kmh',
]

TIME_FEATURES = [
    'target_hour_sin', 'target_hour_cos',
    'target_weekday_sin', 'target_weekday_cos',
    'target_month_sin', 'target_month_cos',
    'target_is_weekend',
]

FEATURES = LOAD_FEATURES + WEATHER_FEATURES + TIME_FEATURES
GROUPS = ['equivalised_income', 'hometype', 'household_size']

IDENTIFIER_COLUMNS = [
    'homeid', 'origin_timestamp_utc', 'target_timestamp_utc',
    'split', 'peak_threshold_w', 'target_power_w', 'target_peak',
    'location',
]

REQUIRED_COLUMNS = set(
    IDENTIFIER_COLUMNS + GROUPS + FEATURES
)

preview = pd.read_csv(PREPARED_DATA_FILE, nrows=5)
available_columns = set(preview.columns)
missing_columns = sorted(REQUIRED_COLUMNS - available_columns)

print(f'Columns found: {len(preview.columns)}')
print(f'Required columns: {len(REQUIRED_COLUMNS)}')

if missing_columns:
    raise ValueError(
        'The prepared file is missing required columns: '
        + ', '.join(missing_columns)
    )

MODEL_FILE = PREPARED_DATA_FILE
print('Validation passed. The file contains all model and weather columns.')
display(preview)


## Cell 10 — Load and validate the prepared modelling table

In [ ]:
def load_selected_columns(path: Path, columns: List[str]) -> pd.DataFrame:
    """Load selected columns from CSV, Parquet or Feather files."""
    name = path.name.lower()

    if name.endswith(".parquet"):
        return pd.read_parquet(path, columns=columns)

    if name.endswith(".feather"):
        data = pd.read_feather(path)
        return data[columns]

    return pd.read_csv(
        path,
        usecols=columns,
        low_memory=False,
    )


def load_modelling_data(path: Path) -> pd.DataFrame:
    """Load, clean and split the prepared modelling dataset."""

    needed_columns = (
        [
            "homeid",
            "origin_timestamp_utc",
            "target_timestamp_utc",
            "split",
            "target_peak",
            "peak_threshold_w",
        ]
        + GROUPS
        + FEATURES
    )

    print("Loading prepared modelling data...")

    data = load_selected_columns(path, needed_columns)

    # Convert timestamps
    data["origin_timestamp_utc"] = pd.to_datetime(
        data["origin_timestamp_utc"],
        errors="coerce",
        utc=True,
    )

    data["target_timestamp_utc"] = pd.to_datetime(
        data["target_timestamp_utc"],
        errors="coerce",
        utc=True,
    )

    # Standardise split labels
    data["split"] = (
        data["split"]
        .astype(str)
        .str.lower()
        .str.strip()
        .replace(
            {
                "val": "validation",
                "valid": "validation",
                "dev": "validation",
            }
        )
    )

    valid_splits = {"train", "validation", "test"}

    unexpected = (
        set(data["split"].dropna().unique())
        - valid_splits
    )

    if unexpected:
        raise ValueError(
            f"Unexpected split labels: {sorted(unexpected)}"
        )

    # Convert the household peak threshold to numeric
    threshold = pd.to_numeric(
        data["peak_threshold_w"],
        errors="coerce",
    )

    # Prevent division by zero
    threshold = threshold.replace(0, np.nan)

    # Scale electricity load features relative to each household threshold
    data[LOAD_FEATURES] = (
        data[LOAD_FEATURES]
        .apply(pd.to_numeric, errors="coerce")
        .div(threshold, axis=0)
    )

    # Convert weather and time features
    numeric_features = WEATHER_FEATURES + TIME_FEATURES

    data[numeric_features] = (
        data[numeric_features]
        .apply(pd.to_numeric, errors="coerce")
    )

    # Reduce memory usage
    data[FEATURES] = data[FEATURES].astype("float32")

    data["target_peak"] = pd.to_numeric(
        data["target_peak"],
        errors="raise",
    ).astype("int8")

    # Clean demographic group columns
    for column in GROUPS:
        data[column] = (
            data[column]
            .fillna("missing")
            .astype(str)
            .str.strip()
        )

    # Remove rows that cannot be used for modelling
    before = len(data)

    data = data.dropna(
        subset=[
            "homeid",
            "split",
            "target_peak",
        ]
        + FEATURES
    ).reset_index(drop=True)

    removed_rows = before - len(data)

    print(
        f"Rows removed because of missing model values: "
        f"{removed_rows:,}"
    )

    # Use smaller samples while testing the notebook
    if DEMO_MODE:
        sampled_parts = []

        for split_name, limit in DEMO_ROWS.items():
            part = data[
                data["split"] == split_name
            ]

            sample_size = min(limit, len(part))

            if sample_size > 0:
                sampled_part = part.sample(
                    n=sample_size,
                    random_state=SEED,
                )

                sampled_parts.append(sampled_part)

        if not sampled_parts:
            raise ValueError(
                "No rows were available for demo sampling."
            )

        data = pd.concat(
            sampled_parts,
            ignore_index=True,
        )

    # Display split information
    split_summary = (
        data.groupby("split")["target_peak"]
        .agg(
            rows="size",
            peaks="sum",
            peak_rate="mean",
        )
    )

    display(split_summary)

    # Confirm all required splits exist
    for split_name in (
        "train",
        "validation",
        "test",
    ):
        if split_name not in split_summary.index:
            raise ValueError(
                f"Missing required split: {split_name}"
            )

    return data


# Load the modelling dataset
if MODEL_FILE is not None:
    data = load_modelling_data(MODEL_FILE)

    train = data[
        data["split"] == "train"
    ].reset_index(drop=True)

    validation = data[
        data["split"] == "validation"
    ].reset_index(drop=True)

    test = data[
        data["split"] == "test"
    ].reset_index(drop=True)

    print(f"Train: {len(train):,}")
    print(f"Validation: {len(validation):,}")
    print(f"Test: {len(test):,}")

else:
    data = None
    train = None
    validation = None
    test = None

    print(
        "Modelling data was not loaded because "
        "no compatible file was found."
    )

## Cell 11 — Basic exploratory analysis

In [ ]:
def run_basic_eda(frame: pd.DataFrame) -> None:
    sns.set_theme(style="whitegrid")

    split_order = ["train", "validation", "test"]

    summary = (
        frame.groupby("split")["target_peak"]
        .agg(rows="size", peaks="sum", peak_rate="mean")
        .reindex(split_order)
    )
    display(summary)

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    for axis, split_name in zip(axes, split_order):
        subset = frame[frame["split"] == split_name]
        sns.countplot(data=subset, x="target_peak", ax=axis)
        axis.set_title(
            f"{split_name.title()}\n"
            f"Peak rate: {subset['target_peak'].mean():.2%}"
        )
        axis.set_xlabel("Target peak")
        axis.set_ylabel("Rows")

    plt.tight_layout()
    plt.show()

    missing = (
        frame[FEATURES + GROUPS]
        .isna()
        .mean()
        .sort_values(ascending=False)
        .rename("missing_rate")
        .to_frame()
    )
    display(missing.head(20))


if data is not None:
    run_basic_eda(data)
else:
    print("Skip: prepared modelling data is unavailable.")

In [ ]:
print("train exists:", "train" in globals())
print("validation exists:", "validation" in globals())
print("test exists:", "test" in globals())

In [ ]:
# Run model training safely
required_variables = [
    "train",
    "validation",
]

missing_variables = [
    variable_name
    for variable_name in required_variables
    if variable_name not in globals()
]

if missing_variables:
    models = None
    validation_scores = None

    print(
        "Training cannot start because these variables "
        f"have not been created: {missing_variables}"
    )

    print(
        "\nRun the modelling-data loading cell first. "
        "That cell should create train, validation and test."
    )

elif train is None or validation is None:
    models = None
    validation_scores = None

    print(
        "Training skipped because train or validation "
        "is currently set to None."
    )

elif train.empty or validation.empty:
    models = None
    validation_scores = None

    print(
        "Training skipped because train or validation "
        "contains no rows."
    )

else:
    models, validation_scores = train_models(
        train,
        validation,
    )

    print("\nTraining complete.")

## Cell 12 — Train Seasonal, XGBoost, and LightGBM models

In [ ]:
from typing import Dict, Tuple

import numpy as np
import pandas as pd


def best_f1_threshold(
    y_true: np.ndarray,
    y_score: np.ndarray,
) -> float:
    """
    Find the probability threshold that gives the highest F1-score
    on the validation dataset.
    """

    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score, dtype=float)

    # Remove missing or invalid values
    valid_mask = (
        np.isfinite(y_true)
        & np.isfinite(y_score)
    )

    y_true = y_true[valid_mask]
    y_score = y_score[valid_mask]

    if len(y_true) == 0:
        raise ValueError(
            "No valid observations were provided for threshold selection."
        )

    if not np.isin(y_true, [0, 1]).all():
        raise ValueError(
            "y_true must contain only binary values: 0 and 1."
        )

    # If there are no positive examples, F1 cannot be optimised
    if y_true.sum() == 0:
        print(
            "Warning: validation data contains no positive examples. "
            "Using threshold 0.5."
        )
        return 0.5

    # Sort scores from highest to lowest
    order = np.argsort(
        -y_score,
        kind="mergesort",
    )

    sorted_y = y_true[order]
    sorted_scores = y_score[order]

    # Calculate cumulative confusion-matrix values
    true_positives = np.cumsum(sorted_y)
    false_positives = np.cumsum(1 - sorted_y)
    false_negatives = sorted_y.sum() - true_positives

    denominator = (
        2 * true_positives
        + false_positives
        + false_negatives
    )

    f1_scores = np.divide(
        2 * true_positives,
        denominator,
        out=np.zeros_like(
            true_positives,
            dtype=float,
        ),
        where=denominator != 0,
    )

    # Only evaluate at the end of each group of equal scores
    final_in_tie = np.concatenate(
        [
            sorted_scores[:-1] != sorted_scores[1:],
            np.array([True]),
        ]
    )

    candidate_indices = np.flatnonzero(final_in_tie)

    if len(candidate_indices) == 0:
        return 0.5

    best_candidate_position = np.argmax(
        f1_scores[candidate_indices]
    )

    best_index = candidate_indices[
        best_candidate_position
    ]

    best_threshold = float(
        sorted_scores[best_index]
    )

    return best_threshold


def train_models(
    train_df: pd.DataFrame,
    validation_df: pd.DataFrame,
) -> Tuple[Dict[str, object], Dict[str, np.ndarray]]:
    """
    Train XGBoost and LightGBM models and generate validation scores.
    """

    if train_df is None or validation_df is None:
        raise ValueError(
            "Training and validation data must be available."
        )

    if train_df.empty:
        raise ValueError(
            "The training dataset is empty."
        )

    if validation_df.empty:
        raise ValueError(
            "The validation dataset is empty."
        )

    missing_train_features = [
        column
        for column in FEATURES
        if column not in train_df.columns
    ]

    missing_validation_features = [
        column
        for column in FEATURES
        if column not in validation_df.columns
    ]

    if missing_train_features:
        raise ValueError(
            "Training data is missing these features: "
            + ", ".join(missing_train_features)
        )

    if missing_validation_features:
        raise ValueError(
            "Validation data is missing these features: "
            + ", ".join(missing_validation_features)
        )

    if "target_peak" not in train_df.columns:
        raise ValueError(
            "The training data does not contain target_peak."
        )

    if "target_peak" not in validation_df.columns:
        raise ValueError(
            "The validation data does not contain target_peak."
        )

    x_train = train_df[FEATURES].copy()
    y_train = (
        train_df["target_peak"]
        .to_numpy(dtype=np.int8)
    )

    x_validation = validation_df[FEATURES].copy()
    y_validation = (
        validation_df["target_peak"]
        .to_numpy(dtype=np.int8)
    )

    # Ensure all model features are numeric
    x_train = x_train.apply(
        pd.to_numeric,
        errors="coerce",
    )

    x_validation = x_validation.apply(
        pd.to_numeric,
        errors="coerce",
    )

    # Check for missing or infinite values
    x_train = x_train.replace(
        [np.inf, -np.inf],
        np.nan,
    )

    x_validation = x_validation.replace(
        [np.inf, -np.inf],
        np.nan,
    )

    train_missing = int(
        x_train.isna().sum().sum()
    )

    validation_missing = int(
        x_validation.isna().sum().sum()
    )

    if train_missing > 0:
        raise ValueError(
            f"Training features contain "
            f"{train_missing:,} missing values."
        )

    if validation_missing > 0:
        raise ValueError(
            f"Validation features contain "
            f"{validation_missing:,} missing values."
        )

    positive_count = int(
        np.sum(y_train == 1)
    )

    negative_count = int(
        np.sum(y_train == 0)
    )

    if positive_count == 0:
        raise ValueError(
            "The training set contains no positive peak examples."
        )

    if negative_count == 0:
        raise ValueError(
            "The training set contains no negative examples."
        )

    imbalance_ratio = (
        negative_count / positive_count
    )

    estimator_count = (
        250 if DEMO_MODE else 600
    )

    print(f"Training rows: {len(x_train):,}")
    print(
        f"Validation rows: "
        f"{len(x_validation):,}"
    )
    print(
        f"Positive training examples: "
        f"{positive_count:,}"
    )
    print(
        f"Negative training examples: "
        f"{negative_count:,}"
    )
    print(
        f"Class imbalance ratio: "
        f"{imbalance_ratio:.2f}"
    )

    models: Dict[str, object] = {
        "XGBoost": XGBClassifier(
            n_estimators=estimator_count,
            max_depth=6,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            min_child_weight=5,
            scale_pos_weight=imbalance_ratio,
            objective="binary:logistic",
            eval_metric="aucpr",
            tree_method="hist",
            n_jobs=-1,
            random_state=SEED,
        ),

        "LightGBM": lgb.LGBMClassifier(
            n_estimators=estimator_count,
            num_leaves=31,
            learning_rate=0.05,
            subsample=0.8,
            subsample_freq=1,
            colsample_bytree=0.8,
            min_child_samples=30,
            scale_pos_weight=imbalance_ratio,
            objective="binary",
            n_jobs=-1,
            random_state=SEED,
            verbosity=-1,
        ),
    }

    if "load_lag_24h_w" not in validation_df.columns:
        raise ValueError(
            "The Seasonal 24h baseline requires "
            "load_lag_24h_w."
        )

    seasonal_scores = (
        pd.to_numeric(
            validation_df["load_lag_24h_w"],
            errors="coerce",
        )
        .fillna(0)
        .to_numpy(dtype=float)
    )

    validation_scores: Dict[str, np.ndarray] = {
        "Seasonal 24h": (
            seasonal_scores >= 1
        ).astype(float)
    }

    for model_name, model in models.items():
        print(f"\nTraining {model_name}...")

        model.fit(
            x_train,
            y_train,
        )

        probabilities = model.predict_proba(
            x_validation
        )[:, 1]

        validation_scores[model_name] = (
            probabilities.astype(float)
        )

        optimal_threshold = best_f1_threshold(
            y_validation,
            probabilities,
        )

        print(
            f"{model_name} completed. "
            f"Best validation F1 threshold: "
            f"{optimal_threshold:.4f}"
        )

    return models, validation_scores


# Run model training
if (
    train is not None
    and validation is not None
):
    models, validation_scores = train_models(
        train,
        validation,
    )

    print("\nTraining complete.")

else:
    models = None
    validation_scores = None

    print(
        "Training skipped because the prepared "
        "modelling data is unavailable."
    )

## Cell 13 — Evaluate models on the test set

In [ ]:
def calculate_metrics(
    y_true: np.ndarray,
    y_score: np.ndarray,
    threshold: float,
) -> Dict:
    y_pred = (y_score >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1],
    ).ravel()

    sensitivity = tp / (tp + fn) if tp + fn else 0.0
    specificity = tn / (tn + fp) if tn + fp else 0.0
    precision = tp / (tp + fp) if tp + fp else 0.0

    f1 = (
        2 * precision * sensitivity / (precision + sensitivity)
        if precision + sensitivity
        else 0.0
    )

    return {
        "roc_auc": roc_auc_score(y_true, y_score),
        "average_precision": average_precision_score(y_true, y_score),
        "f1": f1,
        "precision": precision,
        "sensitivity": sensitivity,
        "specificity": specificity,
        "balanced_accuracy": (sensitivity + specificity) / 2,
        "mcc": matthews_corrcoef(y_true, y_pred),
        "threshold": threshold,
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "prevalence": float(y_true.mean()),
    }


def evaluate_models(
    fitted_models: Dict,
    validation_score_map: Dict,
    validation_df: pd.DataFrame,
    test_df: pd.DataFrame,
) -> Tuple[pd.DataFrame, Dict, Dict]:
    y_validation = validation_df["target_peak"].to_numpy()
    y_test = test_df["target_peak"].to_numpy()

    test_scores = {
        "Seasonal 24h": (
            test_df["load_lag_24h_w"].to_numpy() >= 1
        ).astype(float)
    }

    for model_name, model in fitted_models.items():
        test_scores[model_name] = model.predict_proba(
            test_df[FEATURES]
        )[:, 1]

    rows = []
    predictions = {}

    for model_name, validation_score in validation_score_map.items():
        threshold = best_f1_threshold(
            y_validation,
            validation_score,
        )

        metrics = calculate_metrics(
            y_test,
            test_scores[model_name],
            threshold,
        )

        rows.append({"model": model_name, **metrics})
        predictions[model_name] = (
            test_scores[model_name] >= threshold
        ).astype("int8")

    return pd.DataFrame(rows), predictions, test_scores


if models is not None:
    metrics_df, predictions, test_scores = evaluate_models(
        models,
        validation_scores,
        validation,
        test,
    )

    display(
        metrics_df[
            [
                "model",
                "roc_auc",
                "average_precision",
                "f1",
                "precision",
                "sensitivity",
                "specificity",
                "balanced_accuracy",
                "threshold",
            ]
        ].round(4)
    )
else:
    metrics_df = predictions = test_scores = None
    print("Skip: models were not trained.")

## Cell 14 — Performance plots

In [ ]:
def plot_model_performance(
    test_df: pd.DataFrame,
    score_map: Dict,
    prediction_map: Dict,
) -> None:
    y_true = test_df["target_peak"].to_numpy()

    plt.figure(figsize=(8, 6))
    for model_name, scores in score_map.items():
        fpr, tpr, _ = roc_curve(y_true, scores)
        auc_value = roc_auc_score(y_true, scores)
        plt.plot(fpr, tpr, label=f"{model_name}: {auc_value:.3f}")

    plt.plot([0, 1], [0, 1], linestyle="--")
    plt.xlabel("False-positive rate")
    plt.ylabel("True-positive rate")
    plt.title("ROC curves")
    plt.legend()
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(8, 6))
    for model_name, scores in score_map.items():
        precision, recall, _ = precision_recall_curve(
            y_true,
            scores,
        )
        ap = average_precision_score(y_true, scores)
        plt.plot(recall, precision, label=f"{model_name}: {ap:.3f}")

    plt.axhline(y_true.mean(), linestyle="--", label="Prevalence")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title("Precision-recall curves")
    plt.legend()
    plt.tight_layout()
    plt.show()

    model_count = len(prediction_map)
    fig, axes = plt.subplots(
        1,
        model_count,
        figsize=(5 * model_count, 4),
    )

    if model_count == 1:
        axes = [axes]

    for axis, (model_name, prediction) in zip(
        axes,
        prediction_map.items(),
    ):
        matrix = confusion_matrix(y_true, prediction)
        sns.heatmap(
            matrix,
            annot=True,
            fmt="d",
            cmap="Blues",
            cbar=False,
            ax=axis,
        )
        axis.set_title(model_name)
        axis.set_xlabel("Predicted")
        axis.set_ylabel("Actual")

    plt.tight_layout()
    plt.show()


if test_scores is not None:
    plot_model_performance(test, test_scores, predictions)
else:
    print("Skip: evaluation scores are unavailable.")

## Cell 15 — Demographic fairness audit

In [ ]:
def group_sensitivity(
    test_df: pd.DataFrame,
    prediction_map: Dict,
) -> pd.DataFrame:
    rows = []

    for model_name, model_predictions in prediction_map.items():
        working = test_df[
            ["target_peak"] + GROUPS
        ].copy()
        working["prediction"] = model_predictions

        for attribute in GROUPS:
            for group_name, subset in working.groupby(
                attribute,
                observed=True,
            ):
                positives = subset["target_peak"] == 1
                positive_count = int(positives.sum())

                sensitivity = (
                    subset.loc[positives, "prediction"].mean()
                    if positive_count
                    else np.nan
                )

                rows.append({
                    "model": model_name,
                    "attribute": attribute,
                    "group": str(group_name),
                    "rows": len(subset),
                    "peaks": positive_count,
                    "sensitivity": sensitivity,
                })

    return pd.DataFrame(rows)


def bootstrap_income_gap(
    test_df: pd.DataFrame,
    prediction_map: Dict,
    repeats: int = BOOTSTRAP_REPEATS,
) -> pd.DataFrame:
    rng = np.random.default_rng(SEED)
    rows = []

    for model_name, model_predictions in prediction_map.items():
        frame = test_df[
            ["homeid", "equivalised_income", "target_peak"]
        ].copy()
        frame["prediction"] = model_predictions
        frame["true_positive"] = (
            frame["target_peak"] * frame["prediction"]
        )

        household_totals = (
            frame.groupby(
                ["homeid", "equivalised_income"],
                observed=True,
            )
            .agg(
                peaks=("target_peak", "sum"),
                true_positives=("true_positive", "sum"),
            )
            .reset_index()
        )

        household_totals = household_totals[
            household_totals["equivalised_income"] != "missing"
        ].reset_index(drop=True)

        if household_totals["homeid"].nunique() < 10:
            print(
                f"{model_name}: too few households for a stable bootstrap."
            )
            continue

        bootstrap_gaps = []

        for _ in range(repeats):
            sampled = household_totals.iloc[
                rng.integers(
                    0,
                    len(household_totals),
                    len(household_totals),
                )
            ]

            totals = sampled.groupby(
                "equivalised_income",
                observed=True,
            )[["peaks", "true_positives"]].sum()

            sensitivity = (
                totals["true_positives"]
                .div(totals["peaks"].replace(0, np.nan))
                .dropna()
            )

            if len(sensitivity) >= 2:
                bootstrap_gaps.append(
                    sensitivity.max() - sensitivity.min()
                )

        original_totals = household_totals.groupby(
            "equivalised_income",
            observed=True,
        )[["peaks", "true_positives"]].sum()

        original_sensitivity = (
            original_totals["true_positives"]
            .div(original_totals["peaks"].replace(0, np.nan))
            .dropna()
        )

        if len(original_sensitivity) >= 2 and bootstrap_gaps:
            rows.append({
                "model": model_name,
                "households": household_totals[
                    "homeid"
                ].nunique(),
                "repeats": len(bootstrap_gaps),
                "sensitivity_gap": (
                    original_sensitivity.max()
                    - original_sensitivity.min()
                ),
                "ci_2.5%": np.quantile(
                    bootstrap_gaps, 0.025
                ),
                "ci_97.5%": np.quantile(
                    bootstrap_gaps, 0.975
                ),
            })

    return pd.DataFrame(rows)


if predictions is not None:
    group_results = group_sensitivity(test, predictions)
    income_bootstrap = bootstrap_income_gap(
        test,
        predictions,
    )

    display(group_results.head(30))
    display(income_bootstrap.round(4))

    shown = group_results[
        (group_results["group"] != "missing")
        & group_results["sensitivity"].notna()
    ]

    if not shown.empty:
        chart = sns.catplot(
            data=shown,
            x="group",
            y="sensitivity",
            hue="model",
            col="attribute",
            kind="bar",
            sharex=False,
            height=5,
            aspect=1.1,
        )
        chart.set_xticklabels(rotation=35, ha="right")
        chart.set_axis_labels(
            "",
            "Sensitivity (true-positive rate)",
        )
        chart.set_titles("{col_name}")
        chart.figure.tight_layout()
        plt.show()
else:
    group_results = income_bootstrap = None
    print("Skip: model predictions are unavailable.")

## Cell 16 — Feature importance and save outputs

In [ ]:
def feature_importance_table(fitted_models: Dict) -> pd.DataFrame:
    rows = []

    for model_name, model in fitted_models.items():
        if not hasattr(model, "feature_importances_"):
            continue

        for feature_name, importance in zip(
            FEATURES,
            model.feature_importances_,
        ):
            rows.append({
                "model": model_name,
                "feature": feature_name,
                "importance": float(importance),
            })

    return pd.DataFrame(rows)


if models is not None:
    importance_df = feature_importance_table(models)

    top_features = (
        importance_df.groupby("feature")["importance"]
        .mean()
        .nlargest(15)
        .index
    )

    plt.figure(figsize=(10, 8))
    sns.barplot(
        data=importance_df[
            importance_df["feature"].isin(top_features)
        ],
        y="feature",
        x="importance",
        hue="model",
    )
    plt.title("Top 15 model feature importances")
    plt.tight_layout()
    plt.show()

    metrics_df.to_csv(
        OUTPUT_DIR / "model_metrics.csv",
        index=False,
    )
    group_results.to_csv(
        OUTPUT_DIR / "group_sensitivity.csv",
        index=False,
    )
    income_bootstrap.to_csv(
        OUTPUT_DIR / "income_bootstrap.csv",
        index=False,
    )
    importance_df.to_csv(
        OUTPUT_DIR / "feature_importance.csv",
        index=False,
    )

    print("Saved output files:")
    for path in sorted(OUTPUT_DIR.glob("*")):
        print(" -", path)
else:
    print(
        "No model outputs were saved because a prepared modelling "
        "file was not found."
    )

## more Graph for banner 

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np


def plot_poster_model_comparison(
    metrics_df: pd.DataFrame,
    save_path: str = "poster_model_comparison.png",
) -> None:
    """
    Create a poster-ready comparison of model performance.

    Expected columns in metrics_df:
    model, average_precision, f1, sensitivity,
    specificity and balanced_accuracy.
    """

    required_columns = [
        "model",
        "average_precision",
        "f1",
        "sensitivity",
        "specificity",
        "balanced_accuracy",
    ]

    missing_columns = [
        column
        for column in required_columns
        if column not in metrics_df.columns
    ]

    if missing_columns:
        raise ValueError(
            "metrics_df is missing these columns: "
            + ", ".join(missing_columns)
        )

    plot_data = metrics_df[required_columns].copy()

    metric_labels = {
        "average_precision": "Average precision",
        "f1": "F1-score",
        "sensitivity": "Sensitivity",
        "specificity": "Specificity",
        "balanced_accuracy": "Balanced accuracy",
    }

    metric_columns = list(metric_labels.keys())
    model_names = plot_data["model"].astype(str).tolist()

    x_positions = np.arange(len(metric_columns))
    number_of_models = len(model_names)

    total_group_width = 0.78
    bar_width = total_group_width / number_of_models

    fig, axis = plt.subplots(figsize=(13, 7))

    for model_index, model_name in enumerate(model_names):
        model_values = plot_data.loc[
            plot_data["model"] == model_name,
            metric_columns,
        ].iloc[0].to_numpy(dtype=float)

        bar_positions = (
            x_positions
            - total_group_width / 2
            + bar_width / 2
            + model_index * bar_width
        )

        bars = axis.bar(
            bar_positions,
            model_values,
            width=bar_width,
            label=model_name,
        )

        # Add values above each bar
        for bar, value in zip(bars, model_values):
            axis.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.015,
                f"{value:.2f}",
                ha="center",
                va="bottom",
                fontsize=9,
                rotation=90,
            )

    axis.set_title(
        "Peak-demand prediction performance by model",
        fontsize=18,
        fontweight="bold",
        pad=18,
    )

    axis.set_ylabel(
        "Performance score",
        fontsize=13,
    )

    axis.set_xlabel(
        "Evaluation metric",
        fontsize=13,
    )

    axis.set_xticks(x_positions)

    axis.set_xticklabels(
        [metric_labels[column] for column in metric_columns],
        fontsize=11,
    )

    axis.set_ylim(0, 1.08)

    axis.axhline(
        y=0.5,
        linestyle="--",
        linewidth=1,
        alpha=0.5,
    )

    axis.grid(
        axis="y",
        linestyle="--",
        alpha=0.3,
    )

    axis.legend(
        title="Model",
        frameon=False,
        fontsize=11,
        title_fontsize=11,
        loc="upper center",
        bbox_to_anchor=(0.5, -0.15),
        ncol=max(1, number_of_models),
    )

    axis.spines["top"].set_visible(False)
    axis.spines["right"].set_visible(False)

    fig.text(
        0.5,
        0.01,
        "Thresholds were selected using validation-set F1-score; "
        "performance was evaluated on the held-out test set.",
        ha="center",
        fontsize=10,
    )

    plt.tight_layout(rect=[0, 0.08, 1, 1])

    plt.savefig(
        save_path,
        dpi=300,
        bbox_inches="tight",
    )

    plt.show()

    print(f"Poster graph saved as: {save_path}")


plot_poster_model_comparison(
    metrics_df,
    save_path="poster_model_comparison.png",
)